# 门控 KV 选择的算法规格（教学版）


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 从标准 SDPA 到 Gated-KV

第 10 章的 SDPA 对**全部** KV 做注意力。Gated-KV 的改动是：KV 沿序列切成块（如每块 128 个 token），每块有一个**门控分数** g_j，仅分数通过门槛（top-k 或阈值）的块参与注意力：

$$
\text{sel} = \operatorname{TopK}_{j}(g_j), \qquad
O = \operatorname{softmax}\!\left(\frac{Q K_{\text{sel}}^{\mathrm{T}}}{\sqrt{H}}\right) V_{\text{sel}}
$$

被选块集合 sel 之外的 K/V **完全不进入注意力计算**。省略固定的 B/H 因子时，QKᵀ 和 PV 的工作量按 O(S × k × block) 计算；这里的 k 是选中块数。若 ratio 固定，则 k = ceil((S/block) × ratio)，工作量仍随 S 按 O(S²) 增长，只是常数约缩小到 ratio。不要把固定比例选块称为次二次复杂度；选择器本身的成本另计。


## 2. 门控分数从哪来（本课程取最简）

gym 的 quant_lightning_indexer 基于 query/key 及权重计算相关性并选索引；本章不复现其量化和逐 head 数值契约。教学版取两种可验证方案：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">方案</th><th style="text-align:left;">g_j 定义</th><th style="text-align:left;">特点</th></tr>
<tr><td><b>方案 A（随机+固定 seed）</b></td><td>Host 侧生成随机分数</td><td>聚焦"选择后的执行"，门控来源不引入误差面</td></tr>
<tr><td><b>方案 B（QK 轻量打分）</b></td><td>g_j = mean(Q · k_j)（块内均值或首 token 点积）</td><td>引入一个轻量打分段（可选实现，更接近真实形态）</td></tr>
</table>

教学主线用方案 A，方案 B 作为选做——**先保证选块后的注意力计算正确，再研究真实的门控分数**。方案 B 的 mean(Q · k_j) 只是打分思路：实现前还须明确 Q 的行归约、batch 归约，以及 k_j 取块均值还是首 token；否则无法唯一确定供所有 batch/Q 行共享的 g_j。也应单独核算打分成本，不能仅凭这个写法认定它轻量。


## 3. 教学版完整规格

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项</th><th style="text-align:left;">规格</th></tr>
<tr><td>Q / K / V</td><td>[B, S, H] FP16，H=64/128（单头形态，与实战一一致；S 取 2048/4096），块大小 KV_BLOCK=128</td></tr>
<tr><td>门控分数 g</td><td>[num_kv_blocks] FP32，Host 生成有限值（方案 A），同一组 g 在全部 batch/Q 行共享</td></tr>
<tr><td>选择规则</td><td>top-k：k = ceil(num_blocks × ratio)，ratio ∈ {12.5%, 25%, 50%}（研究超参）</td></tr>
<tr><td>输出</td><td>[B, S, H]，FP16</td></tr>
<tr><td>参考实现</td><td>Host 稳定 argsort 取前 k，按分数降序、同分块号升序输出；gather K/V 后调用 torch SDPA</td></tr>
<tr><td>验收</td><td>逐输出 <code>assert_close(rtol=1e-3, atol=1e-3)</code>（RFC 阈值，沿用实战一；对照同语义 gather 版 torch 参考）</td></tr>
</table>

**与 causal 的组合**：RFC 沿用实战一的无 mask/causal 两档，均须验收。causal 按**原始 token 位置**判断选中且可见，不能对压紧后的 K_sel 直接使用 is_causal=True。某些 Q 行可能没有可见选中 key，本课程约定输出 0。


## 4. 语义要点（写代码前必须想清）

1. **softmax 的归一化范围**：只在被选块上归一化——未选块的贡献不存在，而不是置 0（置 -inf 的掩码写法与"跳过"等价，但跳过省算力，这正是意义所在）；
2. **online 状态按选中块推进**：m/l/acc 只被选中块更新——flash 循环里"跳过"是天然的；
3. **确定性**：统一按“分数降序、同分原始块号升序”输出 sel_idx，Kernel 与 Golden 逐位对齐；不能让 atomic_add 的登记顺序决定遍历顺序。NaN/Inf 分数不属于主线输入。


## 5. 数值算例与确定性 Golden

S=512、KV_BLOCK=128、H=64、ratio=50% 时，g=[0.3,0.8,0.1,0.6] 选块 [1,3]，K_sel/V_sel 各为 [B,256,64]，QKᵀ 为 [B,512,256]。这是小规模说明，正式 S 仍为 2048/4096。

**torch.topk 不保证并列索引稳定**。原分数按块号排列时，`argsort(descending=True, stable=True)` 保留同分项的原始顺序，满足块号小者优先。

以下 Golden 无 gym 运行依赖。q/k/v 为同设备的 [B,S,H] FP16；g_cpu 是按块号排列的 Host FP32 分数。causal 的 mask 保留原始位置，并显式处理零有效列行，避免依赖后端对全遮蔽行的特殊行为。


In [ ]:
import math
import torch.nn.functional as F

def gated_kv_reference(q, k, v, g_cpu, ratio, causal=False, kv_block=128):
    assert q.ndim == 3 and q.shape == k.shape == v.shape
    assert q.dtype == k.dtype == v.dtype == torch.float16
    assert q.device == k.device == v.device
    batch, seq, hidden = q.shape
    assert seq > 0 and seq % kv_block == 0 and hidden in (64, 128)
    assert ratio in (0.125, 0.25, 0.5)
    num_blocks = seq // kv_block
    assert g_cpu.device.type == "cpu" and g_cpu.shape == (num_blocks,)
    assert g_cpu.dtype == torch.float32 and torch.isfinite(g_cpu).all()
    count = math.ceil(num_blocks * ratio)
    sel_host = torch.argsort(g_cpu, descending=True, stable=True)[:count]
    sel = sel_host.to(q.device)
    token_idx = (sel[:, None] * kv_block + torch.arange(kv_block, device=q.device)).reshape(-1)
    k_sel = k.index_select(1, token_idx)
    v_sel = v.index_select(1, token_idx)
    if causal:
        visible = token_idx[None, :] <= torch.arange(seq, device=q.device)[:, None]
        has_visible = visible.any(dim=-1)
        safe_mask = visible.clone()
        safe_mask[~has_visible, 0] = True  # 为全遮蔽行临时提供安全计算路径
        result = F.scaled_dot_product_attention(q, k_sel, v_sel, attn_mask=safe_mask,
                                               dropout_p=0.0, is_causal=False)
        result = result.masked_fill(~has_visible[None, :, None], 0.0)
    else:
        result = F.scaled_dot_product_attention(q, k_sel, v_sel, dropout_p=0.0, is_causal=False)
    return sel_host, result

先检查 sel_idx 的长度、范围、唯一性和逐位一致，再检查输出。至少覆盖全同分、k 边界同分、负分、随机分，以及 causal 下最早被选块不为 0 的情形；最后一种会产生需输出 0 的 Q 行。


### 小算例：选中序号绝不是 causal 坐标

用 8 个 token、块长 2 说明规则。四个分数 [0.1,0.9,0.9,-0.2]，选 2 块；同分按块号升序，sel=[1,2]，原 token_idx=[2,3,4,5]。Q 行 1 没有可见 key，输出 0；Q 行 3 只能见 token 2、3；Q 行 7 见全部四个。

若对压紧 K_sel 使用 is_causal=True，Q 行 0 会错误地看到“压紧列 0”，即原 token 2。正确 mask 必须是 `token_idx[col] <= query_row`。分数相反时 sel 也可能按原位置逆序，online 最大值修正仍须工作，不能假设块号递增。此例块长 2 只用于手算，RFC 实现固定为 128。


## 6. 课后练习

1. （计算）S=4096、KV_BLOCK=128、top 25%：标准 SDPA、Gated-KV 的 QK^T 计算量之比？
2. （判断）"未选块以 -inf 掩码参与计算"与"直接跳过"在非空可见集合且相同数值契约下数学等价，跳过通常有减少工作量的潜力（是否更快需实测）。（对/错）
3. （实践）写出 torch 参考实现（方案 A）：g → 稳定排序取前 k → gather K/V → torch SDPA，作为你后续验收的 golden。

In [ ]:
!cat ./answer/11.02_answer.txt

## 7. 本节小结

- 规格核心：**top-k 选块 + 选中块 SDPA**，注意力工作量 O(S×k×block)；固定 ratio 时仍为 O(S²)，只减少常数；
- 门控来源教学化（方案 A 主线）：先把选择后执行做对；
- 语义三要点：**只对选中块归一化、online 状态只被选中块更新、top-k 确定性破平**。